In [ ]:
import numpy as np
import pandas as pd
from typing import Tuple, Dict
import warnings, os, textwrap
from scipy.stats import mannwhitneyu, ks_2samp
from statsmodels.stats.proportion import proportions_ztest

In [ ]:
def load_and_harmonise(twitch_path: str, youtube_path: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    def _load(fp: str) -> pd.DataFrame:
        ext = os.path.splitext(fp)[1].lower()
        if ext in {'.feather', '.ft'}:
            return pd.read_feather(fp)
        if ext in {'.parquet', '.pq'}:
            return pd.read_parquet(fp)
        return pd.read_csv(fp)

    twitch = _load(twitch_path)
    youtube = _load(youtube_path)

    twitch = twitch.rename(columns={
        'message': 'raw_msg',
        'streamer_name': 'streamer'
    })

    youtube = youtube.rename(columns={
        'message': 'raw_msg',
        'streamer_name': 'streamer',
        'video_id': 'session_id'
    })

    keep_cols = [
        'raw_msg', 'msg_clean', 'author',
        'streamer', 'session_id', 'language',
        'sentiment_label', 'sentiment_score',
        'toxicity_label', 'toxicity_score',
        'toxic'
    ]

    twitch = twitch[keep_cols].copy()
    youtube = youtube[keep_cols].copy()

    twitch['platform'] = 'twitch'
    youtube['platform'] = 'youtube'

    cat_cols = ['sentiment_label', 'toxicity_label', 'language', 'platform', 'streamer']
    for df in (twitch, youtube):
        df['msg_len'] = df['raw_msg'].str.split().str.len()
        df['toxic'] = df['toxic'].astype(bool)

        for c in cat_cols:
            df[c] = df[c].astype('category')

        df[['sentiment_score', 'toxicity_score', 'msg_len']] = df[
            ['sentiment_score', 'toxicity_score', 'msg_len']
        ].apply(pd.to_numeric, errors='coerce')

        df.dropna(
            subset=['sentiment_label', 'toxicity_label', 'sentiment_score', 'toxicity_score'],
            inplace=True
        )

    return twitch, youtube

In [ ]:
twitch_path = 'twitch.csv'
youtube_path = 'youtube.csv'
out_dir = 'results'

In [ ]:
t_df, y_df = load_and_harmonise(twitch_path, youtube_path)
print(f'Twitch rows: {len(t_df):,} | YouTube rows: {len(y_df):,}')

In [ ]:
df = pd.concat([t_df, y_df], ignore_index=True)
df["platform"] = df["platform"].astype("category")
df["language"] = df["language"].astype("category")
df["is_neg"]       = df["sentiment_label"] == "neg"
df["is_toxic"]     = df["toxicity_label"].astype(str).str.endswith("1")
df["neg_or_tox"]   = df["is_neg"] | df["is_toxic"]
df["neg_and_tox"]  = df["is_neg"] & df["is_toxic"]

def prep_bool_counts(data: pd.DataFrame, bool_col: str) -> pd.DataFrame:
    return (
        data.groupby(["language", "platform"], observed=True)[bool_col]
            .agg(k="sum", n="size")
            .reset_index()
    )


def ztest_on_aggregates(tbl: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    rows = []

    overall = (
        tbl.groupby("platform")[["k", "n"]]
        .sum()
        .reindex(["twitch", "youtube"])
    )

    if not overall["k"].isna().any() and not overall["n"].isna().any():
        k_glob, n_glob = overall["k"].to_numpy(), overall["n"].to_numpy()
        z_glob, p_glob = proportions_ztest(k_glob, n_glob)
        prop_t, prop_y = k_glob / n_glob * 100
        diff_glob      = prop_t - prop_y
        h_glob = 2 * (np.arcsin(np.sqrt(k_glob[0] / n_glob[0])) -
                      np.arcsin(np.sqrt(k_glob[1] / n_glob[1])))

        rows.append(dict(
            language      = "overall",
            n_twitch      = n_glob[0], k_twitch = k_glob[0],
            n_youtube     = n_glob[1], k_youtube = k_glob[1],
            prop_twitch   = round(prop_t, 2),
            prop_youtube  = round(prop_y, 2),
            diff_pct      = round(diff_glob, 2),
            z             = z_glob,
            p_value       = p_glob,
            significant   = p_glob < alpha,
            cohens_h      = round(h_glob, 3)
        ))

    for lang, sub in tbl.groupby("language"):
        sub = sub.set_index("platform").reindex(["twitch", "youtube"])
        if sub["k"].isna().any() or sub["n"].isna().any():
            continue

        k, n = sub["k"].to_numpy(), sub["n"].to_numpy()
        z, p = proportions_ztest(k, n)
        prop_t, prop_y = k / n * 100
        diff = prop_t - prop_y
        h = 2 * (np.arcsin(np.sqrt(k[0] / n[0])) -
                 np.arcsin(np.sqrt(k[1] / n[1])))

        rows.append(dict(
            language      = lang,
            n_twitch      = n[0], k_twitch = k[0],
            n_youtube     = n[1], k_youtube = k[1],
            prop_twitch   = round(prop_t, 2),
            prop_youtube  = round(prop_y, 2),
            diff_pct      = round(diff, 2),
            z             = z,
            p_value       = p,
            significant   = p < alpha,
            cohens_h      = round(h, 3)
        ))

    df_res = pd.DataFrame(rows)
    lang_order = (["overall"] +
                  df_res.loc[df_res["language"] != "overall"]
                        .sort_values("diff_pct", ascending=False)["language"].tolist())
    df_res["language"] = pd.Categorical(df_res["language"], categories=lang_order, ordered=True)
    return df_res.sort_values("language").reset_index(drop=True)

bool_tests = {
    col: ztest_on_aggregates(prep_bool_counts(df, col))
    for col in ["is_neg", "is_toxic", "neg_or_tox", "neg_and_tox"]
}


def dist_test(df: pd.DataFrame, score_col: str,
              alpha: float = 0.05, test: str = "mannwhitney") -> pd.DataFrame:
    stat_fun = mannwhitneyu if test == "mannwhitney" else ks_2samp
    rows = []


    t_vals = df.loc[df.platform == "twitch",  score_col].to_numpy()
    y_vals = df.loc[df.platform == "youtube", score_col].to_numpy()
    stat, p = stat_fun(t_vals, y_vals, alternative="two-sided")
    rows.append(dict(
        language      = "overall",
        n_twitch      = len(t_vals),
        n_youtube     = len(y_vals),
        mean_twitch   = t_vals.mean(),
        mean_youtube  = y_vals.mean(),
        diff_mean     = t_vals.mean() - y_vals.mean(),
        stat          = stat,
        p_value       = p,
        significant   = p < alpha,
    ))

    for lang, g in df.groupby("language", observed=True):
        if g["platform"].nunique() < 2:
            continue
        t_vals = g.loc[g.platform == "twitch",  score_col].to_numpy()
        y_vals = g.loc[g.platform == "youtube", score_col].to_numpy()
        if len(t_vals) == 0 or len(y_vals) == 0:
            continue
        stat, p = stat_fun(t_vals, y_vals, alternative="two-sided")
        rows.append(dict(
            language      = lang,
            n_twitch      = len(t_vals),
            n_youtube     = len(y_vals),
            mean_twitch   = t_vals.mean(),
            mean_youtube  = y_vals.mean(),
            diff_mean     = t_vals.mean() - y_vals.mean(),
            stat          = stat,
            p_value       = p,
            significant   = p < alpha,
        ))

    out = pd.DataFrame(rows)
    lang_order = (["overall"] +
                  out.loc[out["language"] != "overall"]
                      .sort_values("diff_mean", ascending=False)["language"].tolist())
    out["language"] = pd.Categorical(out["language"], categories=lang_order, ordered=True)
    return out.sort_values("language").reset_index(drop=True)

sentiment_test = dist_test(df, "sentiment_score", test="mannwhitney")
toxicity_test  = dist_test(df, "toxicity_score",  test="mannwhitney")

In [6]:
sentiment_test

,language,n_twitch,n_youtube,mean_twitch,mean_youtube,diff_mean,stat,p_value,significant
0,overall,2859222,1414199,0.562013,0.664578,-0.102565,1.343076e+12,0.000000e+00,True
1,Chinese,43026,14325,0.542330,0.569738,-0.027408,2.748311e+08,4.557617e-84,True
2,Thai,15117,7546,0.561549,0.624085,-0.062536,4.422720e+07,1.222456e-167,True
3,Portuguese,147395,73696,0.581082,0.653562,-0.072480,4.121193e+09,0.000000e+00,True
4,Arabic,14536,7262,0.577646,0.650270,-0.072624,3.999993e+07,2.982120e-187,True
5,Turkish,10175,5068,0.554083,0.648956,-0.094873,1.771560e+07,4.344070e-218,True
6,Ukranian,10466,5233,0.572225,0.668921,-0.096696,1.856636e+07,5.617331e-238,True
7,Russian,220947,110472,0.563583,0.660356,-0.096773,8.127927e+09,0.000000e+00,True
8,English,1708556,853759,0.556412,0.658033,-0.101621,4.848120e+11,0.000000e+00,True
9,Spanish,240425,112829,0.576711,0.678807,-0.102096,8.997951e+09,0.000000e+00,True


In [7]:
toxicity_test

,language,n_twitch,n_youtube,mean_twitch,mean_youtube,diff_mean,stat,p_value,significant
0,overall,2859222,1414199,0.953494,0.970803,-0.017309,1.483424e+12,0.000000e+00,True
1,Chinese,43026,14325,0.922619,0.896650,0.025969,3.559582e+08,1.355026e-170,True
2,Japanese,121811,60834,0.942755,0.951147,-0.008392,3.067802e+09,0.000000e+00,True
3,Portuguese,147395,73696,0.941918,0.953297,-0.011379,4.532713e+09,0.000000e+00,True
4,Ukranian,10466,5233,0.965767,0.977746,-0.011979,1.958922e+07,1.564471e-186,True
5,Polish,22385,11153,0.952397,0.964606,-0.012209,9.838372e+07,5.029030e-220,True
6,Spanish,240425,112829,0.950539,0.962958,-0.012419,1.080552e+10,0.000000e+00,True
7,Turkish,10175,5068,0.940778,0.954361,-0.013583,2.182429e+07,5.584593e-54,True
8,Thai,15117,7546,0.928689,0.943702,-0.015012,4.452828e+07,6.009283e-160,True
9,Russian,220947,110472,0.956911,0.973652,-0.016741,9.009779e+09,0.000000e+00,True


In [ ]:
res.columns

In [8]:
print("\n=== Z-TEST SU COLONNE BOOLEANE ===")
for name, res in bool_tests.items():
    print(f"\n-- {name} --")
    print(res.head(12))


=== Z-TEST SU COLONNE BOOLEANE ===

-- is_neg --
      language  n_twitch  k_twitch  n_youtube  k_youtube  prop_twitch  \
0      overall   2859222    877411    1414199     523761        30.69   
1       Arabic     14536      3364       7262       1368        23.14   
2     Japanese    121811     37196      60834      16615        30.54   
3      Italian     33433     12631      16707       6273        37.78   
4         Thai     15117      4124       7546       2066        27.28   
5      Russian    220947     58033     110472      31313        26.27   
6       German    132568     36164      66242      20137        27.28   
7       Polish     22385      5977      11153       3344        26.70   
8      Spanish    240425     71715     112829      38809        29.83   
9   Portuguese    147395     45376      73696      26090        30.79   
10     Turkish     10175      2994       5068       1810        29.43   
11     English   1708556    535769     853759     336840        31.36   



# Plots

In [ ]:
sentiment_test["language"] = sentiment_test["language"].replace("overall", "Overall")

row_colors = []
for lang, sig in zip(sentiment_test["language"], sentiment_test["significant"]):
    if lang == "Overall":
        row_colors.append('#f3e8fd') 
    elif sig:
        row_colors.append('white')  
    else:
        row_colors.append('white')    

import plotly.graph_objects as go

fig = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=[
                    "<b>Language</b>", "<b>Twitch Mean</b>", "<b>YouTube Mean</b>",
                    "<b>Diff Mean</b>", "<b>t-Statistic</b>", "<b>p-value</b>", "<b>Significant</b>"
                ],
                fill_color='whitesmoke',
                align='center',
                font=dict(size=12, color='black'),
                line_color='lightgrey',
                height=32
            ),
            cells=dict(
                values=[
                    sentiment_test["language"],
                    sentiment_test["mean_twitch"].round(3),
                    sentiment_test["mean_youtube"].round(3),
                    sentiment_test["diff_mean"].round(3),
                    sentiment_test["stat"].round(2),
                    [f"{p:.2e}" for p in sentiment_test["p_value"]],
                    sentiment_test["significant"]
                ],
                fill_color=[row_colors] * 7,
                align='center',
                font=dict(size=11, color='black'),
                line_color='lightgrey',
                height=28
            )
        )
    ]
)

fig.update_layout(
    title="Sentiment Score Comparison — Twitch vs YouTube by Language",
    title_x=0.5,
    height=700,
    width=950,
    paper_bgcolor="white"
)

fig.show()

In [ ]:
toxicity_test["language"] = toxicity_test["language"].replace("overall", "Overall")

row_colors = []
for lang, sig in zip(toxicity_test["language"], toxicity_test["significant"]):
    if lang == "Overall":
        row_colors.append('#f8d7da')  
    elif sig:
        row_colors.append('white')  
    else:
        row_colors.append('white')    

import plotly.graph_objects as go

fig = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=[
                    "<b>Language</b>", "<b>Twitch Mean</b>", "<b>YouTube Mean</b>",
                    "<b>Diff Mean</b>", "<b>t-Statistic</b>", "<b>p-value</b>", "<b>Significant</b>"
                ],
                fill_color='whitesmoke',
                align='center',
                font=dict(size=12, color='black'),
                line_color='lightgrey',
                height=32
            ),
            cells=dict(
                values=[
                    toxicity_test["language"],
                    toxicity_test["mean_twitch"].round(3),
                    toxicity_test["mean_youtube"].round(3),
                    toxicity_test["diff_mean"].round(3),
                    toxicity_test["stat"].round(2),
                    [f"{p:.2e}" for p in toxicity_test["p_value"]],
                    toxicity_test["significant"]
                ],
                fill_color=[row_colors] * 7,
                align='center',
                font=dict(size=11, color='black'),
                line_color='lightgrey',
                height=28
            )
        )
    ]
)

fig.update_layout(
    title="Toxicity Score Comparison — Twitch vs YouTube by Language",
    title_x=0.5,
    height=700,
    width=950,
    paper_bgcolor="white"
)

fig.show()

In [ ]:
from IPython.display import display, HTML
fmt_bool = {
    "prop_twitch" : "{:.2f}%",
    "prop_youtube": "{:.2f}%",
    "diff_pct"    : "{:+.2f}%",
    "p_value"     : "{:.2e}",
    "cohens_h"    : "{:.3f}",
}

fmt_cont = {
    "mean_twitch" : "{:.3f}",
    "mean_youtube": "{:.3f}",
    "diff_mean"   : "{:+.3f}",
    "p_value"     : "{:.2e}",
}

def highlight_sig(s):
    return ["background-color:#ffeeba" if v else "" for v in s]

def show_df(name: str, df: pd.DataFrame, fmt: dict):
    display(HTML(f"<h3 style='margin:0'>{name}</h3>"))
    sty = (df
           .style
           .format(fmt, na_rep="–")
           .apply(highlight_sig, subset=["significant"]))
    display(sty)

show_df("Sentiment score • Mann-Whitney", sentiment_test, fmt_cont)
show_df("Toxicity score • Mann-Whitney",  toxicity_test,  fmt_cont)

for metric, res in bool_tests.items():
    show_df(f"Z-test su {metric}", res, fmt_bool)


,language,n_twitch,n_youtube,mean_twitch,mean_youtube,diff_mean,stat,p_value,significant
0,overall,2859222,1414199,0.562,0.665,-0.103,1343075703341.500000,0.00e+00,True
1,Chinese,43026,14325,0.542,0.570,-0.027,274831054.000000,4.56e-84,True
2,Thai,15117,7546,0.562,0.624,-0.063,44227202.500000,1.22e-167,True
3,Portuguese,147395,73696,0.581,0.654,-0.072,4121192566.000000,0.00e+00,True
4,Arabic,14536,7262,0.578,0.650,-0.073,39999927.000000,2.98e-187,True
5,Turkish,10175,5068,0.554,0.649,-0.095,17715605.000000,4.34e-218,True
6,Ukranian,10466,5233,0.572,0.669,-0.097,18566357.000000,5.62e-238,True
7,Russian,220947,110472,0.564,0.660,-0.097,8127926821.500000,0.00e+00,True
8,English,1708556,853759,0.556,0.658,-0.102,484811995385.000000,0.00e+00,True
9,Spanish,240425,112829,0.577,0.679,-0.102,8997950713.000000,0.00e+00,True


,language,n_twitch,n_youtube,mean_twitch,mean_youtube,diff_mean,stat,p_value,significant
0,overall,2859222,1414199,0.953,0.971,-0.017,1483424075561.000000,0.00e+00,True
1,Chinese,43026,14325,0.923,0.897,+0.026,355958218.000000,1.36e-170,True
2,Japanese,121811,60834,0.943,0.951,-0.008,3067802496.500000,0.00e+00,True
3,Portuguese,147395,73696,0.942,0.953,-0.011,4532713146.000000,0.00e+00,True
4,Ukranian,10466,5233,0.966,0.978,-0.012,19589220.500000,1.56e-186,True
5,Polish,22385,11153,0.952,0.965,-0.012,98383723.500000,5.03e-220,True
6,Spanish,240425,112829,0.951,0.963,-0.012,10805523460.000000,0.00e+00,True
7,Turkish,10175,5068,0.941,0.954,-0.014,21824289.500000,5.58e-54,True
8,Thai,15117,7546,0.929,0.944,-0.015,44528285.000000,6.01e-160,True
9,Russian,220947,110472,0.957,0.974,-0.017,9009778928.500000,0.00e+00,True


,language,n_twitch,k_twitch,n_youtube,k_youtube,prop_twitch,prop_youtube,diff_pct,z,p_value,significant,cohens_h
0,overall,2859222,877411,1414199,523761,30.69%,37.04%,-6.35%,-131.553825,0.00e+00,True,-0.134
1,Arabic,14536,3364,7262,1368,23.14%,18.84%,+4.30%,7.266400,3.69e-13,True,0.106
2,Japanese,121811,37196,60834,16615,30.54%,27.31%,+3.22%,14.244189,4.87e-46,True,0.071
3,Italian,33433,12631,16707,6273,37.78%,37.55%,+0.23%,0.507221,6.12e-01,False,0.005
4,Thai,15117,4124,7546,2066,27.28%,27.38%,-0.10%,-0.156359,8.76e-01,False,-0.002
5,Russian,220947,58033,110472,31313,26.27%,28.34%,-2.08%,-12.715600,4.84e-37,True,-0.047
6,German,132568,36164,66242,20137,27.28%,30.40%,-3.12%,-14.551877,5.68e-48,True,-0.069
7,Polish,22385,5977,11153,3344,26.70%,29.98%,-3.28%,-6.321156,2.60e-10,True,-0.073
8,Spanish,240425,71715,112829,38809,29.83%,34.40%,-4.57%,-27.300301,4.21e-164,True,-0.098
9,Portuguese,147395,45376,73696,26090,30.79%,35.40%,-4.62%,-21.879953,4.03e-106,True,-0.098


,language,n_twitch,k_twitch,n_youtube,k_youtube,prop_twitch,prop_youtube,diff_pct,z,p_value,significant,cohens_h
0,overall,2859222,448116,1414199,218392,15.67%,15.44%,+0.23%,6.162263,7.17e-10,True,0.006
1,Arabic,14536,2056,7262,637,14.14%,8.77%,+5.37%,11.361720,6.49e-30,True,0.170
2,Russian,220947,48820,110472,19416,22.10%,17.58%,+4.52%,30.338260,3.59e-202,True,0.114
3,Italian,33433,5129,16707,1916,15.34%,11.47%,+3.87%,11.762768,6.07e-32,True,0.114
4,German,132568,16271,66242,6702,12.27%,10.12%,+2.16%,14.175582,1.30e-45,True,0.068
5,Thai,15117,1979,7546,864,13.09%,11.45%,+1.64%,3.515900,4.38e-04,True,0.050
6,Turkish,10175,1058,5068,449,10.40%,8.86%,+1.54%,2.998045,2.72e-03,True,0.052
7,Spanish,240425,37076,112829,16587,15.42%,14.70%,+0.72%,5.558850,2.72e-08,True,0.020
8,English,1708556,264556,853759,134045,15.48%,15.70%,-0.22%,-4.504497,6.65e-06,True,-0.006
9,Portuguese,147395,24468,73696,12410,16.60%,16.84%,-0.24%,-1.421961,1.55e-01,False,-0.006


,language,n_twitch,k_twitch,n_youtube,k_youtube,prop_twitch,prop_youtube,diff_pct,z,p_value,significant,cohens_h
0,overall,2859222,1048895,1414199,584344,36.68%,41.32%,-4.64%,-92.787533,0.00e+00,True,-0.095
1,Arabic,14536,4551,7262,1711,31.31%,23.56%,+7.75%,11.914967,9.89e-33,True,0.174
2,Japanese,121811,43222,60834,19810,35.48%,32.56%,+2.92%,12.366780,3.95e-35,True,0.062
3,Italian,33433,14137,16707,6718,42.28%,40.21%,+2.07%,4.441041,8.95e-06,True,0.042
4,Thai,15117,4885,7546,2362,32.31%,31.30%,+1.01%,1.541368,1.23e-01,False,0.022
5,Russian,220947,78983,110472,39491,35.75%,35.75%,-0.00%,-0.000186,1.00e+00,False,-0.000
6,German,132568,43825,66242,22312,33.06%,33.68%,-0.62%,-2.783636,5.38e-03,True,-0.013
7,Spanish,240425,85710,112829,43682,35.65%,38.72%,-3.07%,-17.634013,1.35e-69,True,-0.063
8,Turkish,10175,3515,5068,1934,34.55%,38.16%,-3.62%,-4.387906,1.14e-05,True,-0.075
9,Portuguese,147395,54838,73696,30406,37.20%,41.26%,-4.05%,-18.461389,4.22e-76,True,-0.083


,language,n_twitch,k_twitch,n_youtube,k_youtube,prop_twitch,prop_youtube,diff_pct,z,p_value,significant,cohens_h
0,overall,2859222,276632,1414199,157809,9.68%,11.16%,-1.48%,-47.761067,0.00e+00,True,-0.049
1,Russian,220947,27870,110472,11238,12.61%,10.17%,+2.44%,20.535333,1.04e-93,True,0.077
2,Italian,33433,3623,16707,1471,10.84%,8.80%,+2.03%,7.098639,1.26e-12,True,0.068
3,Arabic,14536,869,7262,294,5.98%,4.05%,+1.93%,5.975534,2.29e-09,True,0.089
4,Thai,15117,1218,7546,568,8.06%,7.53%,+0.53%,1.395535,1.63e-01,False,0.020
5,German,132568,8610,66242,4527,6.49%,6.83%,-0.34%,-2.870160,4.10e-03,True,-0.014
6,Spanish,240425,23081,112829,11714,9.60%,10.38%,-0.78%,-7.272186,3.54e-13,True,-0.026
7,Portuguese,147395,15006,73696,8094,10.18%,10.98%,-0.80%,-5.812673,6.15e-09,True,-0.026
8,Turkish,10175,537,5068,325,5.28%,6.41%,-1.14%,-2.858405,4.26e-03,True,-0.048
9,Japanese,121811,7028,60834,4471,5.77%,7.35%,-1.58%,-13.102089,3.20e-39,True,-0.064


In [ ]:
res["language"] = res["language"].replace("overall", "Overall")

row_colors = []
for lang, sig in zip(res["language"], res["significant"]):
    if lang == "Overall":
        row_colors.append('#d7ecff')  
    elif sig:
        row_colors.append('white')  
    else:
        row_colors.append('white')   

import plotly.graph_objects as go

fig = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=[
                    "<b>Language</b>", "<b>Twitch %</b>", "<b>YouTube %</b>",
                    "<b>Diff %</b>", "<b>Z-stat</b>", "<b>p-value</b>",
                    "<b>Cohen's h</b>", "<b>Significant</b>"
                ],
                fill_color='whitesmoke',
                align='center',
                font=dict(size=12, color='black'),
                line_color='lightgrey',
                height=32
            ),
            cells=dict(
                values=[
                    res["language"],
                    res["prop_twitch"],
                    res["prop_youtube"],
                    res["diff_pct"],
                    res["z"].round(2),
                    [f"{p:.2e}" for p in res["p_value"]],
                    res["cohens_h"].round(3),
                    res["significant"]
                ],
                fill_color=[row_colors] * 8,
                align='center',
                font=dict(size=11, color='black'),
                line_color='lightgrey',
                height=28
            )
        )
    ]
)

fig.update_layout(
    title="Z-test Proportion Comparison — Twitch vs YouTube by Language",
    title_x=0.5,
    height=700,
    width=1000,
    paper_bgcolor="white"
)

fig.show()

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

def plot_effect_size_res(df, alpha=0.05):
    df_plot = df.copy()

    df_plot['Significant'] = np.where(df_plot['p_value'] < alpha, 'p < 0.05', 'n.s.')
    df_plot['Significant'] = pd.Categorical(df_plot['Significant'], categories=['p < 0.05', 'n.s.'])

    df_plot = df_plot.sort_values('cohens_h', ascending=True)
    df_plot['language'] = pd.Categorical(df_plot['language'], categories=df_plot['language'], ordered=True)

    df_plot['abs_diff'] = df_plot['diff_pct'].abs()

    fig = px.scatter(
        df_plot,
        x='cohens_h',
        y='language',
        color='Significant',
        size='abs_diff',
        size_max=30,
        labels={
            'cohens_h': "Effect Size (Cohen's h)",
            'language': "Language",
            'abs_diff': "Bubble size reflects: |Proportion Difference|",
            'Significant': "Statistical Significance"
        },
        title="Effect Size (Cohen's h) — Twitch vs YouTube by Language",
        color_discrete_map={
            'p < 0.05': '#F5533D',
            'n.s.': '#B6B6B6'
        },
        hover_data={
            'prop_twitch': ':.2f',
            'prop_youtube': ':.2f',
            'diff_pct': ':.2f',
            'z': ':.2f',
            'p_value': ':.3f'
        }
    )

    fig.add_vline(x=0, line_dash="dash", line_color="black")
    fig.update_traces(marker=dict(line=dict(width=1, color="black")))
    fig.update_layout(
        height=650,
        width=1000,
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend_title_text='Statistical Significance',
        legend=dict(itemsizing='constant')
    )
    fig.add_annotation(
    text="Note: Bubble size reflects the absolute value of Proportion Difference",
    xref="paper",
    yref="paper",
    x= 0.65,       
    y=-0.15,        
    showarrow=False,
    font=dict(size=12, color="gray"),
    align="left",
    xanchor="left"
)

    fig.for_each_xaxis(lambda axis: axis.update(showgrid=True, gridcolor='lightgrey'))
    fig.for_each_yaxis(lambda axis: axis.update(showgrid=True, gridcolor='lightgrey'))

    fig.show()

In [ ]:
plot_effect_size_res(res)